# EcoGraphRAG — Notebook 4: Full Graph-RAG

**Goal:** FAISS + Graph BFS merged retrieval + Mistral-7B on 500 Qs.

**Inputs:** `chunks.json`, `hotpotqa_500.json`, `faiss_index.bin`, `entity_graph.gpickle`

**Outputs:** `graph_rag_results.json`

## 1. Setup

In [ ]:
from google.colab import drive
import os, json, time, re, pickle

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/graphrag_research/'
DRIVE_DATA = DRIVE_ROOT + 'data/'
DRIVE_INDICES = DRIVE_ROOT + 'indices/'
DRIVE_OUTPUTS = DRIVE_ROOT + 'outputs/'
DRIVE_CHECKPOINTS = DRIVE_ROOT + 'checkpoints/'

for d in [DRIVE_OUTPUTS, DRIVE_CHECKPOINTS]:
    os.makedirs(d, exist_ok=True)
print('Drive mounted.')

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu spacy networkx
!python -m spacy download en_core_web_sm -q

## 2. Load Data + Indices

In [ ]:
import torch, gc, numpy as np, faiss, spacy
from collections import defaultdict, deque
from sentence_transformers import SentenceTransformer

with open(DRIVE_DATA + 'hotpotqa_500.json') as f:
    questions = json.load(f)['questions']
with open(DRIVE_DATA + 'chunks.json') as f:
    chunks = json.load(f)
chunk_lookup = {c['chunk_id']: c for c in chunks}
chunk_mapping = [c['chunk_id'] for c in chunks]
index = faiss.read_index(DRIVE_INDICES + 'faiss_index.bin')
with open(DRIVE_INDICES + 'entity_graph.gpickle', 'rb') as f:
    graph = pickle.load(f)
nlp = spacy.load('en_core_web_sm', disable=['tagger','parser','lemmatizer'])
print(f'Qs:{len(questions)} Chunks:{len(chunks)} FAISS:{index.ntotal} Graph:{graph.number_of_nodes()}n/{graph.number_of_edges()}e')

## 3. Retrieval Functions

In [ ]:
EMBEDDING_MODEL = 'nomic-ai/nomic-embed-text-v1'
FAISS_TOP_K = 5
GRAPH_TOP_K = 5
BFS_DEPTH = 2
MERGED_TOP_K = 6
GRAPH_WEIGHT = 0.5
FAISS_WEIGHT = 0.5

# === Precompute ALL query embeddings, then free the model ===
embed_model = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True, device='cpu')
print('Encoding all 500 query embeddings (CPU)...')
all_queries = ['search_query: ' + q['question'] for q in questions]
all_query_embeddings = embed_model.encode(all_queries, normalize_embeddings=True, batch_size=64, show_progress_bar=True).astype(np.float32)
print(f'Encoded {len(all_query_embeddings)} queries. Shape: {all_query_embeddings.shape}')

# DELETE the embedding model to free ~1.5GB RAM
del embed_model
gc.collect()
print('Embedding model deleted. Memory freed.')

def faiss_retrieve(question_idx, top_k=FAISS_TOP_K):
    '''Retrieve using precomputed embedding by question index.'''
    q_emb = all_query_embeddings[question_idx:question_idx+1]
    sc, ix = index.search(q_emb, top_k)
    return [dict(chunk_lookup[chunk_mapping[i]], score=float(s)) for s,i in zip(sc[0],ix[0]) if chunk_mapping[i] in chunk_lookup]

def extract_query_entities(question):
    doc = nlp(question)
    seen, ents = set(), []
    for e in doc.ents:
        k = e.text.strip().lower()
        if len(k)>=2 and k not in seen: seen.add(k); ents.append(k)
    return ents

def graph_bfs(q_ents, depth=BFS_DEPTH, top_k=GRAPH_TOP_K):
    visited, cs, nt = set(), defaultdict(float), 0
    for ent in q_ents:
        if ent not in graph: continue
        q, lv = deque([(ent,0)]), set()
        while q:
            nd, d = q.popleft()
            if nd in lv: continue
            lv.add(nd)
            if nd not in visited:
                visited.add(nd); nt += 1
                for cid in graph.nodes[nd].get('chunk_ids',[]):
                    cs[cid] += 1.0/(1.0+d)
            if d < depth:
                for nb in graph.neighbors(nd):
                    if nb not in lv: q.append((nb,d+1))
    ranked = sorted(cs.items(), key=lambda x:x[1], reverse=True)
    return [c for c,_ in ranked[:top_k]], nt

def merged_retrieve(question, question_idx):
    '''Now takes question_idx to use precomputed embeddings.'''
    fr = faiss_retrieve(question_idx)
    qe = extract_query_entities(question)
    gi, nt = graph_bfs(qe)
    scores, sources = defaultdict(float), defaultdict(list)
    if fr:
        mx,mn = max(r['score'] for r in fr), min(r['score'] for r in fr)
        rng = mx-mn if mx!=mn else 1.0
        for r in fr:
            scores[r['chunk_id']] += FAISS_WEIGHT*((r['score']-mn)/rng)
            sources[r['chunk_id']].append('faiss')
    for rank, cid in enumerate(gi):
        scores[cid] += GRAPH_WEIGHT*(1.0/(1.0+rank))
        sources[cid].append('graph')
    ranked = sorted(scores.items(), key=lambda x:x[1], reverse=True)
    merged = []
    for cid,sc in ranked[:MERGED_TOP_K]:
        if cid in chunk_lookup:
            c = dict(chunk_lookup[cid])
            c['merged_score']=round(sc,4); c['source']='+'.join(sources[cid])
            merged.append(c)
    return merged, qe, nt

# Test
tm,te,tn = merged_retrieve(questions[0]['question'], 0)
print(f'Test: {len(tm)} chunks, ents={te}, nodes={tn}')
for c in tm: print(f'  [{c["merged_score"]:.3f}] ({c["source"]}) {c["chunk_id"]}|{c["title"]}')

## 4. Load Mistral-7B

In [ ]:
# Free GPU memory before loading LLM
gc.collect()
torch.cuda.empty_cache()
print(f'GPU before LLM: {torch.cuda.memory_allocated()/(1024**2):.0f} MB')

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
LLM_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'
bnb_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(LLM_MODEL,quantization_config=bnb_config,device_map='auto',torch_dtype=torch.float16)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f'Model loaded. GPU: {torch.cuda.max_memory_allocated()/(1024**2):.0f} MB')

In [ ]:
PROMPT_TEMPLATE = '<s>[INST] Answer the following question using ONLY the provided context.\nGive a short, precise answer (1-5 words). Do not explain.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer: [/INST]'

def generate_answer(question, ctx_chunks):
    ctx = '\n\n'.join([c['text'] for c in ctx_chunks])
    prompt = PROMPT_TEMPLATE.format(context=ctx, question=question)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)
    inputs = {k:v.to(model.device) for k,v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs,max_new_tokens=64,do_sample=False,temperature=0.1,pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

## 5. Run Graph-RAG (500 Qs)

In [ ]:
CHECKPOINT_FILE = DRIVE_CHECKPOINTS+'graph_rag_checkpoint.json'
RESULTS_FILE = DRIVE_OUTPUTS+'graph_rag_results.json'

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f: ckpt=json.load(f)
    results,start_idx = ckpt['results'],ckpt['current_index']+1
    print(f'Resuming: {len(results)} results, idx {start_idx}')
else: results,start_idx = [],0

total = len(questions)
print(f'Running Graph-RAG: {start_idx} to {total-1}...\n')
for i in range(start_idx, total):
    q = questions[i]
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    merged, q_ents, nodes_t = merged_retrieve(q['question'], i)
    pred = generate_answer(q['question'], merged)
    lat = time.time()-t0
    gpu = torch.cuda.max_memory_allocated()/(1024**2)
    results.append({'id':q['id'],'question':q['question'],'prediction':pred,'gold':q['answer'],'type':q['type'],'level':q.get('level','unknown'),'latency_seconds':round(lat,3),'peak_gpu_mb':round(gpu,2),'chunks_retrieved':len(merged),'nodes_traversed':nodes_t,'query_entities':q_ents,'sources':[c.get('source','') for c in merged]})
    if (i+1)%10==0 or i==total-1:
        print(f'  [{i+1}/{total}] {lat:.2f}s nodes={nodes_t} Pred:"{pred[:35]}" Gold:"{q["answer"]}"')
    if (i+1)%25==0:
        with open(CHECKPOINT_FILE,'w') as f: json.dump({'current_index':i,'results':results},f,ensure_ascii=False)
        print(f'  Checkpoint saved')
        gc.collect(); torch.cuda.empty_cache()  # periodic cleanup

with open(RESULTS_FILE,'w') as f: json.dump(results,f,indent=2,ensure_ascii=False)
print(f'\nSaved {len(results)} results')
if os.path.exists(CHECKPOINT_FILE): os.remove(CHECKPOINT_FILE)

## 6. Evaluation

In [ ]:
import string
from collections import Counter
def normalize(s):
    s=s.lower(); s=re.sub(r'\b(a|an|the)\b',' ',s); s=''.join(c for c in s if c not in string.punctuation); return ' '.join(s.split())
def em(p,g): return int(normalize(p)==normalize(g))
def f1(p,g):
    pt,gt=normalize(p).split(),normalize(g).split()
    if not pt or not gt: return float(pt==gt)
    c=sum((Counter(pt)&Counter(gt)).values())
    if c==0: return 0.0
    return 2*(c/len(pt))*(c/len(gt))/(c/len(pt)+c/len(gt))

em_s=[em(r['prediction'],r['gold']) for r in results]
f1_s=[f1(r['prediction'],r['gold']) for r in results]
print(f'\n{"="*60}\nGRAPH-RAG RESULTS ({len(results)} Qs)\n{"="*60}')
print(f'Overall EM: {sum(em_s)/len(em_s):.4f} ({sum(em_s)/len(em_s)*100:.1f}%)')
print(f'Overall F1: {sum(f1_s)/len(f1_s):.4f} ({sum(f1_s)/len(f1_s)*100:.1f}%)')
for qt in ['bridge','comparison']:
    tr=[r for r in results if r['type']==qt]
    if tr:
        te=[em(r['prediction'],r['gold']) for r in tr]; tf=[f1(r['prediction'],r['gold']) for r in tr]
        print(f'\n{qt.capitalize()} ({len(tr)}): EM:{sum(te)/len(te):.4f} F1:{sum(tf)/len(tf):.4f}')
nl=[r['nodes_traversed'] for r in results]; la=[r['latency_seconds'] for r in results]; gm=[r['peak_gpu_mb'] for r in results]
print(f'\nNodes: avg={sum(nl)/len(nl):.0f} max={max(nl)}')
print(f'Latency: avg={sum(la)/len(la):.2f}s')
print(f'GPU: max={max(gm):.0f} MB')
metrics={'experiment':'graph_rag','num_questions':len(results),'overall_em':round(sum(em_s)/len(em_s),4),'overall_f1':round(sum(f1_s)/len(f1_s),4),'avg_latency_s':round(sum(la)/len(la),3),'max_gpu_mb':round(max(gm),2),'avg_nodes_traversed':round(sum(nl)/len(nl),1)}
with open(DRIVE_OUTPUTS+'graph_rag_metrics.json','w') as f: json.dump(metrics,f,indent=2)
print('\nMetrics saved. Graph-RAG complete!')